<a href="https://colab.research.google.com/github/mrkewenn/EE-Llama-Tuner/blob/main/LLMs_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLMs Fine Tuning Project

In [ ]:
# Primeiros passos, instalando biblioteca Unsloth
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.7/842.7 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
# Carregando modelo base e quantize (QLoRA)
from unsloth import FastLanguageModel

In [ ]:
# Carregando modelo e Tonkenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-bnb-4bit", # Pode ser alterado para qualquer modelo 7B do Unsloth
    max_seq_length = 2048, # Maxima sequencia que o modelo pode suportar
    dtype = None, # None deixa o auto-detection abilitado
    # load_in_4bit = True, # Aplica 4-bit quantization para salvar VRAM e permitir rodar no Colab
)

# Preparando modelo para PEFT
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Define o rank das matrizes do LoRA, quanto maior o rank melhor e mais uso de VRAM
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
        ], # Camadas especificas da rede em que estaremos injetando os adaptadores
    lora_alpha = 128, # Fator de escala (comumente 2x rank)
    lora_dropout = 0, # Zero define "no regularization", fazendo o modelo determinista e otimizado
    bias = "none", # Aplicando LoRA apenas a pesos
    use_gradient_checkpointing = "unsloth", # Reduzindo ainda mais o uso de VRAM
)

==((====))==  Unsloth 2026.5.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-2-7b-bnb-4bit as a legacy tokenizer.


Preparando o Dataset e Divisao 80-20

In [ ]:
from datasets import load_dataset

# Carregando dataset
ds = load_dataset("STEM-AI-mtl/Electrical-engineering")
main_ds = ds["train"]

# Configurando a divisao 80-20
split_ds = main_ds.train_test_split(test_size = 0.2)
train_data = split_ds["train"]
eval_data = split_ds["test"]

# Definindo o template do formato de prompt do Llama 2 seguindo os padroes da documentacao da Meta
llama_template = """<s>[INST] <<SYS>>
You are a brilliant and highly accurate Electrical Engineering assistant.
<</SYS>>

{user_prompt} [/INST] {model_response} </s>"""

# Funcao para mapear nosso dataset para o template do Llama 2
def format_prompts(examples):
  instructions = examples["instruction"]
  inputs = examples["input"]
  outputs = examples["output"]

  texts = []
  for inst, inp, out in zip(instructions, inputs, outputs):
    # Combinamos instruction com input devido ao modo que o Llama funciona(espera apenas um user_prompt)
    # Se o input estiver vazio, ele vai utilizar a instrucao
    if inp and str(inp).strip() != "":
      combined_prompt = f"{inst}\n\nContext/Data:\n{inp}"
    else:
      combined_prompt = inst


    prompt = llama_template.format(user_prompt = combined_prompt, model_response = out)
    texts.append(prompt)

  return { "text" : texts }

# Aplicando o formato para o dataset de treino e teste
train_data = train_data.map(format_prompts, batched = True)
eval_data = eval_data.map(format_prompts, batched = True)

print(f"Training records: {len(train_data)}")
print(f"Testing records: {len(eval_data)}")

Map:   0%|          | 0/904 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

Training records: 904
Testing records: 227


Testando Modelo-Base

In [ ]:
# Carregando o modelo de inferencia do Unsloth, que e consideravelmente rapido
FastLanguageModel.for_inference(model)

# Fazer um questionamento ligeiramente complexo dentro do contexto de Engenharia Eletrica
question = "Explain the fundamental difference between active and passive components in an electrical circuit, and provide two examples of each."

# Formatar para que continue de acordo com o modelo que configuramos anteriormente
test_prompt = f"""<s>[INST] <<SYS>>
You are a brilliant and highly accurate Electrical Engineering assistant.
<</SYS>>

{question} [/INST]"""

# "Tokenizando" o input e mandando pra GPU
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# Gera a resposta, max_tokens define o tamanho da resposta e use_cache acelera geracao
outputs = model.generate(**inputs, max_new_tokens=256, use_cache = True)

# Decodifica os tokens em texto legivel e printa
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print("--- BASE MODEL RESPONSE ---")
print(decoded_output)

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

--- BASE MODEL RESPONSE ---
['[INST] <<SYS>>\nYou are a brilliant and highly accurate Electrical Engineering assistant.\n<</SYS>>\n\nExplain the fundamental difference between active and passive components in an electrical circuit, and provide two examples of each. [/INST]\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</SYS>>\n\n<</']


Iniciando processo de Fine Tuning Supervisionado (SFT)

In [ ]:
# Utilizando o pacote otimizado do proprio Unsloth para SFT, SFTTrainer para treinar nosso modelo
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bf16_supported
import torch

# Configurando SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = eval_data,
    dataset_text_field = "text",
    packing = True,
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2, # Quantidade de sequencias que a GPU vai fazer leitura
        gradient_accumulation_steps = 4, # Numero de iteracoes acumuladas antes de aplicar otimizador
        warmup_steps = 5, # Gradualmente aumenta a taxa de aprendizado
        max_steps = 60, # Limite de etapas de treino, 60 para esse teste
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1, # Log metrica de loss
        optim = "adamw_8bit", # 8-bit para salvar ram
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
)
)

# Checando status da GPU e CPU antes de processo
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved before training.")

# Iniciando Fine Tuning
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/904 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/227 [00:00<?, ? examples/s]

GPU = Tesla T4. Max memory = 14.563 GB.
7.824 GB of memory reserved before training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 904 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 159,907,840 of 6,898,323,456 (2.32% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.577302
2,2.726797
3,2.268313
4,2.018164
5,1.663063
6,1.465894
7,1.066773
8,1.003012
9,0.916727
10,0.777152


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


Testando modelo fine tuned

In [ ]:
# Carregando o modelo de inferencia do Unsloth, que e consideravelmente rapido
FastLanguageModel.for_inference(model)

# Usamos exatamente a mesma pergunta
question = "Explain the fundamental difference between active and passive components in an electrical circuit, and provide two examples of each."

test_prompt = f"""<s>[INST] <<SYS>>
You are a brilliant and highly accurate Electrical Engineering assistant.
<</SYS>>

{question} [/INST]"""

# Tokenizando e gerando a nova resposta
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, use_cache = True)
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print("--- FINE TUNED MODEL RESPONSE ---")
print(decoded_output)

# Salvando meu modelo
model.save_pretrained("lora_electrical_engineering")
tokenizer.save_pretrained("lora_electrical_engineering")

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


--- FINE TUNED MODEL RESPONSE ---
['[INST] <<SYS>>\nYou are a brilliant and highly accurate Electrical Engineering assistant.\n<</SYS>>\n\nExplain the fundamental difference between active and passive components in an electrical circuit, and provide two examples of each. [/INST] Active components, such as transistors, diodes, and ICs, are capable of amplifying or switching signals. They require an external power source and are used for signal processing, control, and switching. Passive components, like resistors, capacitors, and inductors, only allow the flow of current and do not require an external power source. They are used for filtering, impedance matching, and stabilization. ']


Unsloth: Restored added_tokens_decoder metadata in lora_electrical_engineering/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in lora_electrical_engineering.


('lora_electrical_engineering/tokenizer_config.json',
 'lora_electrical_engineering/tokenizer.json')